# B3b (v2) — Burgers Representation Test

**Status of this experiment:** boundary/confound-elimination, NOT mechanistic. It tests whether the Panda-vs-Chronos MAE advantage on Burgers survives a change in spatial-to-channel encoding. It does not probe Panda's internals and cannot explain *why* Panda wins in cases where it does. See discussion in chat before treating any "robust" verdict as an explanation.

**Non-redundancy note.** Experiment 12 already compared PCA vs. Uniform vs. Stratified-Uniform vs. Diversity subsampling at $\nu \in \{0.05, 0.005\}$ and found the advantage direction survives across all of them (though magnitude differs — PCA advantage $\approx 3\times$ the spatial-subsample advantage at $\nu=0.05$). Re-running PCA/Subsample at $\nu=0.05$ would just reproduce that. So:
- **$\nu = 1.0$**: run PCA, Subsample, Fourier fresh. This regime was never tested for representation dependence, and it's the one A3 actually implicates (near-dead PCA channels, failed eDMD).
- **$\nu = 0.05$**: run **Fourier only** fresh. PCA and Subsample(Stratified) values are **cited** from Experiment 12 (`fixed_subsampling_results.csv`), not re-simulated.

**Added diagnostic (cheap, no model calls):** for every (ν, representation) cell, compute effective rank (participation ratio of singular values) and near-dead-channel count of the *input data matrix itself*, mirroring A3's approach. This tests whether the channel-degeneracy problem A3 found is PCA-specific or generic to 16-channel compression of this field — an input-side check, not a Panda-internals check.

**Pre-registered decision rule (fixed before running), MAE arm:**
- At a given ν, advantage is **REPRESENTATION-ROBUST** only if significant ($p<0.05$) and same-signed under both alternative representations (fresh-run or cited) relative to PCA.
- 1-of-2 survival → **MIXED**. 0-of-2 → **PCA-SPECIFIC**.

**Pre-registered decision rule, channel-health diagnostic:**
- A representation is flagged **DEGENERATE** at a given ν if effective rank $< C/2 = 8$ or near-dead-channel count (variance $<1\%$ of that matrix's max-channel variance) $\geq 4$ of 16. Both thresholds are chosen defaults, not literature-derived — report as such, don't treat as a hard scientific boundary.

**Consistency gate:** recomputed PCA arm at $\nu=1.0$ must match Experiment 10's logged value (adv $=+0.0382$, $p=0.0039$).

## Cell 1 — PASTE-IN PLACEHOLDER
Paste your real harness (imports, panda_model/chronos_model load, CONTEXT_LEN, instance_norm_window, load_ts, panda_forecast, chronos_forecast, evaluate) from new_experiments.ipynb. Not reconstructed from memory.

In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd
from scipy.stats import wilcoxon, linregress
from scipy.integrate import solve_ivp
from sklearn.metrics import pairwise_distances
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 8
CONTEXT_LEN = 512
PRED_LEN    = 96
DATA_DIR    = './ts_data'  # adjust if needed

Device: cpu


In [2]:
import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path='GilpinLab/panda',
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print('Models loaded.')

Models loaded.


In [3]:
# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))

# -------------------------------------------------------
# Per-window normalisation
# -------------------------------------------------------
def instance_norm_window(x_CT):
    """x_CT: (C, T). Normalise per channel using this window only."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

def load_ts(path):
    """Raw (C, T) — no global normalisation."""
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)

# -------------------------------------------------------
# Inference
# -------------------------------------------------------
def panda_forecast(context_np, horizon):
    """context_np: (C, T) normalised. Returns (C, horizon)."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)  # (C, horizon)

def chronos_forecast(context_np, horizon):
    """Batched — all channels in one call."""
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)

# -------------------------------------------------------
# Core evaluator
# -------------------------------------------------------
def evaluate(data_CT, horizon, n_windows=N_WINDOWS, label='',
             fn_a=None, fn_b=None,
             name_a='panda', name_b='chronos'):
    """
    data_CT: (C, T) RAW. Normalises each window independently.
    fn_a, fn_b: (context_normed: (C,T), horizon) -> (C, H)
    """
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: T={T} too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        _, pval = wilcoxon(diff, alternative='greater') \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    sig = ' *' if pval < 0.05 else (' ~' if pval < 0.10 else '')
    iqr_a = np.percentile(mae_a,75) - np.percentile(mae_a,25)
    iqr_b = np.percentile(mae_b,75) - np.percentile(mae_b,25)

    result = {
        'label'         : label,
        'horizon'       : horizon,
        'name_a'        : name_a,
        'name_b'        : name_b,
        f'{name_a}_mae' : np.median(mae_a),
        f'{name_a}_iqr' : iqr_a,
        f'{name_b}_mae' : np.median(mae_b),
        f'{name_b}_iqr' : iqr_b,
        'advantage_mae' : adv,
        'wilcoxon_p'    : pval,
    }
    print(
        f'  {label:50s}  H={horizon:4d}  '
        f'{name_a}={np.median(mae_a):.4f}[±{iqr_a:.4f}]  '
        f'{name_b}={np.median(mae_b):.4f}[±{iqr_b:.4f}]  '
        f'Adv={adv:+.4f}  p={pval:.3f}{sig}'
    )
    return result

print('Helpers defined.')

Helpers defined.


## Cell 2 — Burgers solver (verbatim, matches Experiment 10 / fixed_burgers_results.csv)

In [4]:
import numpy as np
import pandas as pd
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd

def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=42):
    rng       = np.random.default_rng(seed)
    dx        = 2 * np.pi / N_x
    dt_diff   = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv    = 0.4 * dx
    dt        = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub     = max(1, int(np.ceil(dt_record / dt)))
    dt_act    = dt_record / n_sub

    k       = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op    = -nu * k**2

    u0_hat = np.zeros(N_x, dtype=complex)
    rng2   = np.random.default_rng(seed)
    for m in range(1, 6):
        amp = rng2.standard_normal() + 1j * rng2.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias

    def rhs(u_hat):
        u  = np.real(ifft(u_hat))
        nl = fft(0.5 * u**2) * dealias
        return L_op * u_hat - 1j * k * nl

    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1    = rhs(u_hat)
            k2    = rhs(u_hat + 0.5*dt_act*k1)
            k3    = rhs(u_hat + 0.5*dt_act*k2)
            k4    = rhs(u_hat +     dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1 + 2*k2 + 2*k3 + k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                print(f"    Diverged at t={t}")
                return U[:t]
    return U

print("Solver check:")
for nu_t in [1.0, 0.05]:
    U_test = simulate_burgers_stable(T=50, nu=nu_t)
    ok = "OK" if len(U_test) == 50 else f"FAILED at step {len(U_test)}"
    print(f"  nu={nu_t}: shape={U_test.shape}, range=[{U_test.min():.3f},{U_test.max():.3f}]  {ok}")

Solver check:
  nu=1.0: shape=(50, 128), range=[-0.060,0.063]  OK
  nu=0.05: shape=(50, 128), range=[-0.060,0.063]  OK


## Cell 3 — Representations + channel-health diagnostic

In [6]:
def pca_reduction(U, n_components):
    """U: (T, N_x) raw. Returns (T, n_components). Verbatim from Experiment 10."""
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape) - 1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)


def spatial_subsample_stratified(U, n_points=16, variance_floor_pct=10):
    """Variance-stratified uniform subsampling (Experiment 12 fix)."""
    variances = U.var(axis=0)
    threshold = np.percentile(variances, variance_floor_pct)
    valid_idx = np.where(variances >= threshold)[0]
    if len(valid_idx) < n_points:
        valid_idx = np.arange(U.shape[1])
    selected = valid_idx[np.linspace(0, len(valid_idx) - 1, n_points, dtype=int)]
    return U[:, selected].astype(np.float32)


def fourier_reduction(U, n_channels=16):
    """Real+imag of lowest n_channels//2 non-DC wavenumbers. Phase retained."""
    T, N_x  = U.shape
    n_modes = n_channels // 2
    U_hat   = np.fft.rfft(U, axis=1)
    modes   = U_hat[:, 1:n_modes + 1]
    channels = np.empty((T, 2 * n_modes), dtype=np.float32)
    channels[:, 0::2] = modes.real
    channels[:, 1::2] = modes.imag
    return channels.astype(np.float32)


def channel_health_diagnostic(data_TC, dead_var_frac=0.01, rank_frac=0.5):
    """Input-side diagnostic, no model calls.

    effective_rank (participation ratio of singular values) is approximately
    basis-invariant -- it measures the intrinsic dimensionality of the
    underlying trajectory, so it's comparable ACROSS representations at
    fixed underlying dynamics (PCA vs. Subsample vs. Fourier of the SAME U).

    n_dead_channels is NOT basis-invariant and is reported as descriptive
    only -- it does NOT feed the degeneracy verdict. PCA is variance-ordered
    by construction (that's what PCA does), so its trailing channels are
    mechanically low-variance regardless of the true complexity of the
    field. A high n_dead count for PCA vs. low for Subsample/Fourier at
    IDENTICAL effective_rank is expected structurally, not a finding about
    Burgers -- confirmed empirically in the Cell 3 sanity check (eff_rank
    identical at 1.2/16 across all three representations while n_dead
    varied 15/1/9).

    data_TC: (T, C). Returns dict with effective_rank, n_dead_channels,
    degenerate (eff_rank-only verdict).
    """
    T, C = data_TC.shape
    s = svd(data_TC - data_TC.mean(axis=0, keepdims=True), full_matrices=False, compute_uv=False)
    eff_rank = float((s.sum() ** 2) / (np.sum(s ** 2) + 1e-12))

    var_per_channel = data_TC.var(axis=0)
    max_var = var_per_channel.max() + 1e-12
    n_dead = int(np.sum(var_per_channel < dead_var_frac * max_var))

    # Verdict uses effective_rank ONLY -- the basis-comparable measure.
    # n_dead_channels is retained in the output for reference but does not
    # determine "degenerate": it's structurally biased toward flagging
    # variance-ordered representations (PCA) regardless of true complexity.
    degenerate = eff_rank < rank_frac * C
    return {
        "effective_rank": eff_rank,
        "n_dead_channels": n_dead,  # descriptive only -- see docstring
        "n_channels": C,
        "degenerate": degenerate,
    }


# Sanity check
U_check = simulate_burgers_stable(T=200, nu=0.05)
for name, fn in [("PCA", pca_reduction), ("Subsample", spatial_subsample_stratified), ("Fourier", fourier_reduction)]:
    out = fn(U_check, 16)
    diag = channel_health_diagnostic(out)
    print(f"  {name:10s}: shape={out.shape}, std={out.std():.4f}, "
          f"eff_rank={diag['effective_rank']:.1f}/{diag['n_channels']}, "
          f"n_dead={diag['n_dead_channels']}")

  PCA       : shape=(200, 16), std=0.0149, eff_rank=1.2/16, n_dead=15
  Subsample : shape=(200, 16), std=0.0245, eff_rank=1.2/16, n_dead=1
  Fourier   : shape=(200, 16), std=0.5348, eff_rank=1.2/16, n_dead=9


## Cell 4 — Cited baseline (Experiment 12, NOT rerun)

In [7]:
# Cited verbatim from fixed_subsampling_results.csv / Experiment 12 (nu=0.05 rows).
# NOT recomputed in this notebook -- avoids redundant reruns of an already-answered comparison.
cited_exp12_nu05 = {
    "PCA":       {"panda_mae": 0.1374, "chronos_mae": 0.2915, "wilcoxon_p": 0.00390625, "source": "Experiment 12 (cited)"},
    "Subsample": {"panda_mae": 0.0296, "chronos_mae": 0.0736, "wilcoxon_p": 0.00390625, "source": "Experiment 12 (cited, Stratified variant)"},
}
for rep, row in cited_exp12_nu05.items():
    row["advantage_mae_recomputed"] = row["chronos_mae"] - row["panda_mae"]
    row["relative_skill"] = row["chronos_mae"] / row["panda_mae"]
print("Cited Experiment 12 values (nu=0.05):")
for rep, row in cited_exp12_nu05.items():
    print(f"  {rep:10s}: adv={row['advantage_mae_recomputed']:+.4f}  rel_skill={row['relative_skill']:.2f}  p={row['wilcoxon_p']:.4f}")

Cited Experiment 12 values (nu=0.05):
  PCA       : adv=+0.1541  rel_skill=2.12  p=0.0039
  Subsample : adv=+0.0440  rel_skill=2.49  p=0.0039


## Cell 5 — Fresh runs: nu=1.0 (all 3 arms) + nu=0.05 (Fourier only)

In [8]:
N_CH = 16
SEED = 42
p_col_candidates = ["wilcoxon_p", "wilcoxon_p_mae", "p"]

def get_p(res):
    return next((res[c] for c in p_col_candidates if c in res and pd.notna(res[c])), np.nan)

fresh_results = []
U_by_nu = {}

run_plan = {
    1.0:  ["PCA", "Subsample", "Fourier"],
    0.05: ["Fourier"],
}

rep_fns = {"PCA": pca_reduction, "Subsample": spatial_subsample_stratified, "Fourier": fourier_reduction}

for nu, reps_to_run in run_plan.items():
    print(f"\n{'='*70}\nnu = {nu}  (running: {reps_to_run})\n{'='*70}")
    U = simulate_burgers_stable(T=1000, N_x=128, nu=nu, seed=SEED)
    if len(U) < CONTEXT_LEN + 128 + 10:
        print(f"  SKIP nu={nu}: solver produced only {len(U)} steps")
        continue
    U_by_nu[nu] = U

    for rep_name in reps_to_run:
        data_TC = rep_fns[rep_name](U, N_CH)
        diag    = channel_health_diagnostic(data_TC)
        data_CT = data_TC.T
        res = evaluate(data_CT, 128, n_windows=8, label=f"Burgers_{rep_name}_nu={nu:.3f}")
        if res is None:
            continue
        if len(fresh_results) == 0:
            print("\n  [check] evaluate() returned keys:", sorted(res.keys()))
        res["nu"]             = nu
        res["representation"] = rep_name
        res["source"]         = "fresh"
        res["advantage_mae_recomputed"] = res["chronos_mae"] - res["panda_mae"]
        res["relative_skill"] = res["chronos_mae"] / res["panda_mae"] if res["panda_mae"] > 0 else np.nan
        res["wilcoxon_p_resolved"] = get_p(res)
        res.update({f"diag_{k}": v for k, v in diag.items()})
        fresh_results.append(res)

df_fresh = pd.DataFrame(fresh_results)
df_fresh.to_csv("b3b_representation_results.csv", index=False)
print("\nSaved b3b_representation_results.csv")
df_fresh


nu = 1.0  (running: ['PCA', 'Subsample', 'Fourier'])


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=1.000                                H= 128  panda=0.0188[±0.0065]  chronos=0.0650[±0.0593]  Adv=+0.0461  p=0.004 *

  [check] evaluate() returned keys: ['advantage_mae', 'chronos_iqr', 'chronos_mae', 'horizon', 'label', 'name_a', 'name_b', 'panda_iqr', 'panda_mae', 'wilcoxon_p']


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_Subsample_nu=1.000                          H= 128  panda=0.0235[±0.0020]  chronos=0.0481[±0.0489]  Adv=+0.0246  p=0.020 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_Fourier_nu=1.000                            H= 128  panda=0.0162[±0.0170]  chronos=0.0139[±0.0211]  Adv=-0.0023  p=0.727

nu = 0.05  (running: ['Fourier'])


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_Fourier_nu=0.050                            H= 128  panda=0.0259[±0.0019]  chronos=0.0839[±0.1009]  Adv=+0.0580  p=0.004 *

Saved b3b_representation_results.csv


,label,horizon,name_a,name_b,panda_mae,panda_iqr,chronos_mae,chronos_iqr,advantage_mae,wilcoxon_p,nu,representation,source,advantage_mae_recomputed,relative_skill,wilcoxon_p_resolved,diag_effective_rank,diag_n_dead_channels,diag_n_channels,diag_degenerate
0,Burgers_PCA_nu=1.000,128,panda,chronos,0.018828,0.006461,0.064976,0.059297,0.046148,0.003906,1.00,PCA,fresh,0.046148,3.451040,0.003906,2.111827,13,16,True
1,Burgers_Subsample_nu=1.000,128,panda,chronos,0.023507,0.001989,0.048090,0.048888,0.024583,0.019531,1.00,Subsample,fresh,0.024583,2.045813,0.019531,2.070698,0,16,True
2,Burgers_Fourier_nu=1.000,128,panda,chronos,0.016244,0.017048,0.013934,0.021075,-0.002310,0.726562,1.00,Fourier,fresh,-0.002310,0.857791,0.726562,2.111816,9,16,True
3,Burgers_Fourier_nu=0.050,128,panda,chronos,0.025869,0.001871,0.083877,0.100851,0.058008,0.003906,0.05,Fourier,fresh,0.058008,3.242342,0.003906,1.423185,9,16,True


## Cell 6 — Consistency gate (nu=1.0 PCA arm vs. Experiment 10)

In [9]:
logged_exp10_nu10 = {"adv": 0.038243952207267284, "p": 0.00390625}

row = df_fresh[(df_fresh["nu"] == 1.0) & (df_fresh["representation"] == "PCA")]
print("Consistency gate: recomputed PCA @ nu=1.0 vs. logged Experiment 10")
print("-" * 70)
if len(row) == 0:
    print("  No PCA row at nu=1.0 -- did Cell 5 run the full nu=1.0 plan?")
    gate_pass = False
else:
    row  = row.iloc[0]
    adv  = row["advantage_mae_recomputed"]
    diff = adv - logged_exp10_nu10["adv"]
    gate_pass = abs(diff) < 0.01
    print(f"  recomputed adv={adv:+.4f}  logged={logged_exp10_nu10['adv']:+.4f}  diff={diff:+.4f}  "
          f"p={row['wilcoxon_p_resolved']:.4f} (logged p={logged_exp10_nu10['p']:.4f})")
print(f"\nGate: {'PASS' if gate_pass else 'FAIL -- investigate before trusting Subsample/Fourier arms'}")

Consistency gate: recomputed PCA @ nu=1.0 vs. logged Experiment 10
----------------------------------------------------------------------
  recomputed adv=+0.0461  logged=+0.0382  diff=+0.0079  p=0.0039 (logged p=0.0039)

Gate: PASS


## Cell 7 — Unified table + categorical verdicts

In [12]:
# Assemble one unified table across fresh + cited rows for both nu values
rows = []

for _, r in df_fresh.iterrows():
    rows.append({
        "nu": r["nu"], "representation": r["representation"], "source": "fresh",
        "advantage": r["advantage_mae_recomputed"], "relative_skill": r["relative_skill"],
        "p": r["wilcoxon_p_resolved"],
        "eff_rank": r.get("diag_effective_rank", np.nan),
        "n_dead": r.get("diag_n_dead_channels", np.nan),
        "degenerate": r.get("diag_degenerate", np.nan),
    })

for rep, r in cited_exp12_nu05.items():
    # recompute the cheap diagnostic for the cited arms too, using the SAME simulated
    # U at nu=0.05 from this run, so the diagnostic (not the MAE) is still fresh.
    if 0.05 in U_by_nu:
        diag = channel_health_diagnostic(rep_fns[rep](U_by_nu[0.05], N_CH))
    else:
        diag = {"effective_rank": np.nan, "n_dead_channels": np.nan, "degenerate": np.nan}
    rows.append({
        "nu": 0.05, "representation": rep, "source": r["source"],
        "advantage": r["advantage_mae_recomputed"], "relative_skill": r["relative_skill"],
        "p": r["wilcoxon_p"],
        "eff_rank": diag["effective_rank"], "n_dead": diag["n_dead_channels"], "degenerate": diag["degenerate"],
    })

df_unified = pd.DataFrame(rows).sort_values(["nu", "representation"]).reset_index(drop=True)
df_unified["significant_positive"] = (df_unified["p"] < 0.05) & (df_unified["advantage"] > 0)
df_unified.to_csv("b3b_unified_results.csv", index=False)

print("Unified B3b table")
print("-" * 100)
print(df_unified.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n" + "=" * 100)
print("CATEGORICAL VERDICTS")
print("=" * 100)

mae_verdicts = {}
for nu in [1.0, 0.05]:
    sub = df_unified[df_unified["nu"] == nu]
    if len(sub) == 0:
        continue
    pca_row = sub[sub["representation"] == "PCA"]
    alt_survive = sub[(sub["representation"] != "PCA")]["significant_positive"]
    n_survive = int(alt_survive.sum())
    n_total   = len(alt_survive)
    if n_survive == n_total and n_total > 0:
        verdict = "REPRESENTATION-ROBUST"
    elif n_survive == 0:
        verdict = "PCA-SPECIFIC"
    else:
        verdict = f"MIXED ({n_survive}/{n_total} alt. representations survive)"
    mae_verdicts[nu] = verdict
    print(f"\nnu={nu}  MAE-ADVANTAGE VERDICT: {verdict}")
    for _, r in sub.iterrows():
        print(f"    {r['representation']:10s} [{r['source']:5s}]  adv={r['advantage']:+.4f}  "
              f"rel_skill={r['relative_skill']:.2f}  p={r['p']:.4f}  "
              f"{'SIG+' if r['significant_positive'] else 'not sig/not positive'}")

print("\n" + "-" * 100)
print("CHANNEL-HEALTH VERDICT (input-side diagnostic, NOT a Panda-internals probe)")
print("-" * 100)
print("  NOTE: verdict below uses effective_rank ONLY (basis-comparable across")
print("  representations). n_dead_channels is shown for reference but excluded from")
print("  the verdict -- it is structurally biased toward flagging PCA (variance-ordered")
print("  by construction) regardless of the field's true complexity. See Cell 3 docstring.")
for nu in [1.0, 0.05]:
    sub = df_unified[df_unified["nu"] == nu]
    if len(sub) == 0:
        continue
    deg = sub[sub["degenerate"] == True]["representation"].tolist()
    healthy = sub[sub["degenerate"] == False]["representation"].tolist()
    print(f"\n  nu={nu}:")
    for _, r in sub.iterrows():
        print(f"    {r['representation']:10s}  eff_rank={r['eff_rank']:.1f}/16  "
              f"n_dead={r['n_dead']:.0f} (descriptive only)  "
              f"{'DEGENERATE' if r['degenerate'] else 'healthy'}")
    print(f"    --> degenerate={deg or 'none'}   healthy={healthy or 'none'}")
    if "PCA" in deg and len(deg) == 1:
        print("    --> A3's dead-channel finding appears PCA-specific at this nu by effective_rank")
        print("        (input-side only; not re-tested inside Panda).")
    elif len(deg) == len(sub):
        print("    --> All representations degenerate at this nu by effective_rank -- 16 channels")
        print("        may just be too few for this field, independent of method.")
    elif len(deg) == 0:
        print("    --> No representation flagged degenerate at this nu by effective_rank.")
    else:
        print("    --> Mixed -- inspect eff_rank values above directly rather than relying on the label.")

print("\n" + "-" * 100)
print("CROSS-CHECK: does MAE advantage track channel degeneracy, or are they independent?")
print("-" * 100)
corr_check = df_unified.dropna(subset=["advantage", "eff_rank"])
if len(corr_check) >= 3:
    from scipy.stats import spearmanr
    rho, p_corr = spearmanr(corr_check["advantage"], corr_check["eff_rank"])
    print(f"  Spearman(advantage, effective_rank) across {len(corr_check)} cells: rho={rho:.2f}, p={p_corr:.3f}")
    print("  (n is small -- 5-6 cells -- treat this as descriptive, not a powered test.)")
else:
    print("  Not enough cells for a correlation check.")


Unified B3b table
----------------------------------------------------------------------------------------------------
   nu representation                                    source  advantage  relative_skill     p  eff_rank  n_dead  degenerate  significant_positive
0.050        Fourier                                     fresh      0.058           3.242 0.004     1.423       9        True                  True
0.050            PCA                     Experiment 12 (cited)      0.154           2.122 0.004     1.424      14        True                  True
0.050      Subsample Experiment 12 (cited, Stratified variant)      0.044           2.486 0.004     1.422       0        True                  True
1.000        Fourier                                     fresh     -0.002           0.858 0.727     2.112       9        True                 False
1.000            PCA                                     fresh      0.046           3.451 0.004     2.112      13        True                